In [1]:
import os

from pathlib import Path
from typing import (
    Optional, 
    Dict,
    Any,
    List, 
    Tuple,
)

import pandas as pd
import xarray as xr
import numpy  as np

from tqdm import tqdm
from logger import get_logger
from scipy import stats as scipy_stats
from concurrent.futures import ProcessPoolExecutor, as_completed

from constants import (
    BARRA_CATALOGUE,
    EAR5_CATALOGUE,
    ERA5L_CATALOGUE,
    CREATEIP_CATALOGUE,
    BARRA_VARIABLES
)

logger = get_logger()


QUANTILES = [
    0.01, 0.02, 0.03, 0.04, 0.05,
    0.25, 0.50, 0.75,
    0.95, 0.96, 0.97, 0.98, 0.99
]
MAX_WORKERS = 48


In [2]:
%%time
BARRA = pd.read_csv(BARRA_CATALOGUE, compression='gzip')

BARRA = BARRA[BARRA.source_id == 'BARRA-RE2']
BARRA = BARRA[BARRA.domain_id == 'AUS-22']
BARRA = BARRA[BARRA.freq == 'day']
BARRA = BARRA[BARRA.file_type == 'f']
BARRA = BARRA[BARRA.variable_id.isin(BARRA_VARIABLES)]

# Convert start_time safely: 200501.0 -> 200501 -> datetime
BARRA['start_time'] = pd.to_numeric(
    BARRA['start_time'],
    errors='coerce'
).astype('Int64')

BARRA['start_time'] = pd.to_datetime(
    BARRA['start_time'].astype(str),
    format='%Y%m',
    errors='coerce'
)

# Remove failed dates, if any
BARRA = BARRA.dropna(subset=['start_time'])

# Select last 20 calendar years available
latest_year = BARRA['start_time'].dt.year.max()
start_year = latest_year - 35

BARRA = BARRA[
    BARRA['start_time'].dt.year.between(start_year, latest_year)
]

# Convert back to YYYY-MM
BARRA['start_time'] = BARRA['start_time'].dt.strftime('%Y-%m')

BARRA = (
    BARRA[[
    'variable_id',
    'start_time',
    'path'
    ]]
    .sort_values(by='start_time')
    .reset_index(drop=True)
)

CPU times: user 8.4 s, sys: 734 ms, total: 9.14 s
Wall time: 9.13 s


In [3]:
def extract_metadata_from_netcdf(file_path):
    try:
        with xr.open_dataset(file_path, decode_times=True, chunks={}) as ds:
            dates = []

            if 'time' in ds:
                time_values = ds['time'].values
                dates = [
                    pd.Timestamp(t).strftime('%Y-%m-%d')
                    for t in time_values
                ]

            data_vars = [v for v in ds.data_vars if v not in ds.coords]

            if data_vars:
                var_name = data_vars[0]
                dims = {
                    k: v
                    for k, v in ds[var_name].sizes.items()
                    if k != 'time'
                }
                dimensions = str(dims)
            else:
                dimensions = None

            return {
                'path': file_path,
                'dates': dates,
                'dimensions': dimensions
            }

    except Exception as e:
        return {
            'path': file_path,
            'dates': [],
            'dimensions': None,
            'error': str(e)
        }


def extract_dates_concurrent(
    catalogue: pd.DataFrame,
    max_workers= None,
) -> pd.DataFrame:

    if max_workers is None:
        max_workers = min(8, os.cpu_count() or 1)

    catalogue = catalogue.copy()

    # Remove old date columns if present
    cols_to_drop = ['start_date', 'end_date', 'date', 'dimensions']
    catalogue = catalogue.drop(
        columns=[c for c in cols_to_drop if c in catalogue.columns]
    )

    if 'path' not in catalogue.columns:
        raise ValueError("catalogue must contain a 'path' column")

    file_paths = catalogue['path'].dropna().unique().tolist()
    total_files = len(file_paths)

    logger.info(
        f"Extracting metadata from {total_files} NetCDF files "
        f"using {max_workers} workers..."
    )

    results = {}

    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        future_to_path = {
            executor.submit(extract_metadata_from_netcdf, path): path
            for path in file_paths
        }

        with tqdm(
            total=total_files,
            desc="Extracting metadata",
            unit="file"
        ) as pbar:
            for future in as_completed(future_to_path):
                result = future.result()
                results[result['path']] = result
                pbar.update(1)

    expanded_rows = []

    for _, row in catalogue.iterrows():
        path = row['path']
        metadata = results.get(
            path,
            {
                'dates': [],
                'dimensions': None
            }
        )

        dates = metadata.get('dates', [])
        dimensions = metadata.get('dimensions')

        if dates:
            for date in dates:
                new_row = row.to_dict()
                new_row['date'] = date
                new_row['dimensions'] = dimensions
                expanded_rows.append(new_row)
        else:
            new_row = row.to_dict()
            new_row['date'] = None
            new_row['dimensions'] = dimensions
            expanded_rows.append(new_row)

    expanded_df = pd.DataFrame(expanded_rows)

    successful = expanded_df['date'].notna().sum()

    logger.info(
        f"Expanded catalogue: {len(catalogue)} files -> "
        f"{len(expanded_df)} rows ({successful} with dates)"
    )

    return expanded_df

def compute_stats_for_array(data: np.ndarray) -> Optional[Dict[str, float]]:

    data = np.asarray(data, dtype=np.float64).ravel()
    data = data[np.isfinite(data)]

    if data.size == 0:
        return None

    stats_dict = {
        "min": float(np.min(data)),
        "max": float(np.max(data)),
        "mean": float(np.mean(data)),
        "std": float(np.std(data)),
        "median": float(np.median(data)),
        "skewness": float(scipy_stats.skew(data, nan_policy="omit")),
    }

    quantile_values = np.quantile(data, QUANTILES)

    for q, val in zip(QUANTILES, quantile_values):
        stats_dict[f"q{q:.2f}"] = float(val)

    return stats_dict


def extract_stats_from_file_group(
    args: Tuple[str, str, List[str]]
) -> List[Dict[str, Any]]:
    
    file_path, variable_id, date_list = args
    results = []

    try:
        with xr.open_dataset(file_path, decode_times=True) as ds:

            if variable_id not in ds.data_vars:
                for date_str in date_list:
                    results.append({
                        "path": file_path,
                        "variable_id": variable_id,
                        "date": date_str,
                        "stats": None,
                        "error": f"{variable_id} not found in file"
                    })
                return results

            da = ds[variable_id]

            if "time" in da.dims:
                times = pd.to_datetime(ds["time"].values)
                time_dates = np.array([
                    pd.Timestamp(t).strftime("%Y-%m-%d")
                    for t in times
                ])

                for date_str in date_list:
                    date_str = pd.Timestamp(date_str).strftime("%Y-%m-%d")

                    idx = np.where(time_dates == date_str)[0]

                    if len(idx) == 0:
                        results.append({
                            "path": file_path,
                            "variable_id": variable_id,
                            "date": date_str,
                            "stats": None,
                            "error": "date not found in file"
                        })
                        continue

                    data_slice = da.isel(time=idx).values
                    stats = compute_stats_for_array(data_slice)

                    results.append({
                        "path": file_path,
                        "variable_id": variable_id,
                        "date": date_str,
                        "stats": stats,
                        "error": None
                    })

            else:
                data_slice = da.values
                stats = compute_stats_for_array(data_slice)

                for date_str in date_list:
                    date_str = pd.Timestamp(date_str).strftime("%Y-%m-%d")

                    results.append({
                        "path": file_path,
                        "variable_id": variable_id,
                        "date": date_str,
                        "stats": stats,
                        "error": None
                    })

    except Exception as e:
        for date_str in date_list:
            results.append({
                "path": file_path,
                "variable_id": variable_id,
                "date": pd.Timestamp(date_str).strftime("%Y-%m-%d"),
                "stats": None,
                "error": str(e)
            })

    return results


def extract_stats_concurrent(
    catalogue: pd.DataFrame,
    expand_stats: bool = True,
    max_workers = None 
) -> pd.DataFrame:

    if max_workers is None:
        max_workers = min(8, os.cpu_count() or 1)
    
    required_cols = {"path", "variable_id", "date"}
    missing_cols = required_cols - set(catalogue.columns)

    if missing_cols:
        raise ValueError(f"catalogue is missing columns: {missing_cols}")

    catalogue = catalogue.copy()

    # Standardise date format
    catalogue["date"] = pd.to_datetime(
        catalogue["date"],
        errors="coerce"
    ).dt.strftime("%Y-%m-%d")

    # Remove rows without valid dates
    catalogue_valid = catalogue.dropna(subset=["date"]).copy()

    # Group by file and variable so each NetCDF file is opened once
    grouped = (
        catalogue_valid
        .groupby(["path", "variable_id"])["date"]
        .apply(lambda x: sorted(x.dropna().unique().tolist()))
        .reset_index()
    )

    tasks = [
        (row["path"], row["variable_id"], row["date"])
        for _, row in grouped.iterrows()
    ]

    total_tasks = len(tasks)

    logger.info(
        f"Extracting stats from {total_tasks} file-variable groups "
        f"using {max_workers} workers..."
    )

    all_results = []

    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        future_to_task = {
            executor.submit(extract_stats_from_file_group, task): task
            for task in tasks
        }

        with tqdm(
            total=total_tasks,
            desc="Extracting stats",
            unit="file-var"
        ) as pbar:
            for future in as_completed(future_to_task):
                result_list = future.result()
                all_results.extend(result_list)
                pbar.update(1)

    results_df = pd.DataFrame(all_results)

    # Merge stats back onto original BARRA_DAILY catalogue
    output = catalogue.merge(
        results_df,
        on=["path", "variable_id", "date"],
        how="left"
    )

    successful = output["stats"].notna().sum()

    logger.info(
        f"Stats extracted: {successful}/{len(output)} rows"
    )

    if expand_stats:
        stats_expanded = pd.json_normalize(output["stats"])

        output = pd.concat(
            [
                output.drop(columns=["stats"]),
                stats_expanded
            ],
            axis=1
        )

    return output


def extract_global_stats(
    df: pd.DataFrame,
    variables: Optional[List[str]] = None,
) -> Dict[str, Dict[str, float]]:
    """
    Extract global statistics per variable from a flat stats DataFrame.

    Expected columns:
    variable_id, min, max, skewness, q0.98, q0.95
    """

    if variables is None:
        variables = BARRA_VARIABLES

    required_cols = [
        "variable_id",
        "min",
        "max",
        "skewness",
        "q0.98",
        "q0.95",
    ]

    missing_cols = [c for c in required_cols if c not in df.columns]

    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")

    work = df.copy()

    # Ensure stats columns are numeric
    stat_cols = ["min", "max", "skewness", "q0.98", "q0.95"]

    for col in stat_cols:
        work[col] = pd.to_numeric(work[col], errors="coerce")

    global_stats = {}

    for var in variables:
        sub = work[work["variable_id"] == var]

        values = {
            "_min": sub["min"].min(skipna=True) if not sub.empty else None,
            "_max": sub["max"].max(skipna=True) if not sub.empty else None,
            "_skew": sub["skewness"].max(skipna=True) if not sub.empty else None,
            "_q98": sub["q0.98"].min(skipna=True) if not sub.empty else None,
            "_q95": sub["q0.95"].min(skipna=True) if not sub.empty else None,
        }

        # Convert NaN to None and numpy floats to Python floats
        global_stats[var] = {
            key: None if pd.isna(value) else float(value)
            for key, value in values.items()
        }

    return global_stats
    

In [4]:
%%time
BARRA_DAILY = extract_dates_concurrent(
    catalogue=BARRA,
    max_workers=MAX_WORKERS
)
BARRA_DAILY.groupby('variable_id')['date'].agg(['min', 'max', 'count'])


2026-06-16 16:38:26 | INFO    | 1845811415:extract_dates_concurrent:L63 - Extracting metadata from 2120 NetCDF files using 48 workers...
Extracting metadata: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2120/2120 [00:02<00:00, 1046.17file/s]
2026-06-16 16:38:30 | INFO    | 1845811415:extract_dates_concurrent:L117 - Expanded catalogue: 2120 files -> 64520 rows (64520 with dates)


CPU times: user 1.12 s, sys: 992 ms, total: 2.11 s
Wall time: 3.86 s


,min,max,count
variable_id,,,
psl,1990-01-01,2025-04-30,12904
ua100m,1990-01-01,2025-04-30,12904
va100m,1990-01-01,2025-04-30,12904
zg500,1990-01-01,2025-04-30,12904
zmla,1990-01-01,2025-04-30,12904


In [5]:
%%time
stats_path = Path("resources/barra-re-daily-stats.csv.gz")

if stats_path.exists():
    print(f"Loading existing stats: {stats_path}")
    BARRA_DAILY_STATS = pd.read_csv(stats_path, compression="gzip")
    GLOBAL_STATS = extract_global_stats(BARRA_DAILY_STATS)
else:
    print("Stats file not found. Computing stats...")

    BARRA_DAILY_STATS = extract_stats_concurrent(
        catalogue=BARRA_DAILY,
        expand_stats=True,
        max_workers=MAX_WORKERS,
    )

    stats_path.parent.mkdir(parents=True, exist_ok=True)

    BARRA_DAILY_STATS.to_csv(
        stats_path,
        index=False,
        compression="gzip"
    )

    GLOBAL_STATS = extract_global_stats(BARRA_DAILY_STATS)
    print(f"Saved stats to: {stats_path}")
    

Loading existing stats: resources/barra-re-daily-stats.csv.gz
CPU times: user 487 ms, sys: 12.7 ms, total: 500 ms
Wall time: 498 ms


In [6]:
GLOBAL_STATS

{'ua100m': {'_min': -40.46923828125,
  '_max': 45.640869140625,
  '_skew': 1.346128286773972,
  '_q98': 9.042724609375,
  '_q95': 7.413818359375},
 'va100m': {'_min': -41.316650390625,
  '_max': 41.2646484375,
  '_skew': 0.7754697916259109,
  '_q98': 5.476806640625,
  '_q95': 4.386962890625},
 'psl': {'_min': 93401.5625,
  '_max': 104783.0625,
  '_skew': 1.4122856812249351,
  '_q98': 101394.125,
  '_q95': 101251.0625},
 'zg500': {'_min': 4660.125,
  '_max': 5975.528645833333,
  '_skew': -0.5261496382191959,
  '_q98': 5834.881510416667,
  '_q95': 5831.702473958333},
 'zmla': {'_min': 30.453125,
  '_max': 5508.96875,
  '_skew': 2.038968155306477,
  '_q98': 1034.40625,
  '_q95': 943.296875}}

In [8]:
from pathlib import Path
import os
import shutil
import numpy as np
import pandas as pd
import xarray as xr

from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

SCRATCH_DIR = Path("/scratch/ng72/ha2606")
SCRATCH_DIR.mkdir(parents=True, exist_ok=True)

TMP_DIR = SCRATCH_DIR / "tmp"
TMP_DIR.mkdir(parents=True, exist_ok=True)

os.environ["TMPDIR"] = str(TMP_DIR)
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

MONTHLY_DIR = SCRATCH_DIR / "barra_re2_wind100_monthly_nc"
MONTHLY_DIR.mkdir(parents=True, exist_ok=True)

FINAL_NC = SCRATCH_DIR / "barra-re2-wind100-spatial-daily.nc"

FORCE_REFRESH_MONTHLY = False
FORCE_REFRESH_FINAL = True

# Do not use too many workers. Each worker opens two large NetCDF files.
MAX_WORKERS_WIND = min(MAX_WORKERS, 32)

print(f"Monthly output directory: {MONTHLY_DIR}")
print(f"Final NetCDF: {FINAL_NC}")
print(f"Scratch free space: {shutil.disk_usage(SCRATCH_DIR).free / 1e9:.1f} GB")
print(f"Workers: {MAX_WORKERS_WIND}")

Monthly output directory: /scratch/ng72/ha2606/barra_re2_wind100_monthly_nc
Final NetCDF: /scratch/ng72/ha2606/barra-re2-wind100-spatial-daily.nc
Scratch free space: 4776686.0 GB
Workers: 32


In [9]:
BARRA_WIND = BARRA[
    BARRA["variable_id"].isin(["ua100m", "va100m"])
].copy()

BARRA_WIND["start_time"] = BARRA_WIND["start_time"].astype(str)

wind_pairs = (
    BARRA_WIND[["start_time", "variable_id", "path"]]
    .drop_duplicates()
    .pivot_table(
        index="start_time",
        columns="variable_id",
        values="path",
        aggfunc="first"
    )
    .dropna(subset=["ua100m", "va100m"])
    .reset_index()
    .sort_values("start_time")
    .reset_index(drop=True)
)

wind_pairs["out_path"] = wind_pairs["start_time"].apply(
    lambda x: str(MONTHLY_DIR / f"wind100_{x}.nc")
)

print(f"Number of paired months: {len(wind_pairs)}")
display(wind_pairs.head())
display(wind_pairs.tail())

Number of paired months: 424


variable_id,start_time,ua100m,va100m,out_path
0,1990-01,/g/data/ob53/BARRA2/output/reanalysis/AUS-22/B...,/g/data/ob53/BARRA2/output/reanalysis/AUS-22/B...,/scratch/ng72/ha2606/barra_re2_wind100_monthly...
1,1990-02,/g/data/ob53/BARRA2/output/reanalysis/AUS-22/B...,/g/data/ob53/BARRA2/output/reanalysis/AUS-22/B...,/scratch/ng72/ha2606/barra_re2_wind100_monthly...
2,1990-03,/g/data/ob53/BARRA2/output/reanalysis/AUS-22/B...,/g/data/ob53/BARRA2/output/reanalysis/AUS-22/B...,/scratch/ng72/ha2606/barra_re2_wind100_monthly...
3,1990-04,/g/data/ob53/BARRA2/output/reanalysis/AUS-22/B...,/g/data/ob53/BARRA2/output/reanalysis/AUS-22/B...,/scratch/ng72/ha2606/barra_re2_wind100_monthly...
4,1990-05,/g/data/ob53/BARRA2/output/reanalysis/AUS-22/B...,/g/data/ob53/BARRA2/output/reanalysis/AUS-22/B...,/scratch/ng72/ha2606/barra_re2_wind100_monthly...


variable_id,start_time,ua100m,va100m,out_path
419,2024-12,/g/data/ob53/BARRA2/output/reanalysis/AUS-22/B...,/g/data/ob53/BARRA2/output/reanalysis/AUS-22/B...,/scratch/ng72/ha2606/barra_re2_wind100_monthly...
420,2025-01,/g/data/ob53/BARRA2/output/reanalysis/AUS-22/B...,/g/data/ob53/BARRA2/output/reanalysis/AUS-22/B...,/scratch/ng72/ha2606/barra_re2_wind100_monthly...
421,2025-02,/g/data/ob53/BARRA2/output/reanalysis/AUS-22/B...,/g/data/ob53/BARRA2/output/reanalysis/AUS-22/B...,/scratch/ng72/ha2606/barra_re2_wind100_monthly...
422,2025-03,/g/data/ob53/BARRA2/output/reanalysis/AUS-22/B...,/g/data/ob53/BARRA2/output/reanalysis/AUS-22/B...,/scratch/ng72/ha2606/barra_re2_wind100_monthly...
423,2025-04,/g/data/ob53/BARRA2/output/reanalysis/AUS-22/B...,/g/data/ob53/BARRA2/output/reanalysis/AUS-22/B...,/scratch/ng72/ha2606/barra_re2_wind100_monthly...


In [10]:
def process_month_to_netcdf(task):
    """
    Worker function:
    open one monthly ua100m and va100m file,
    compute wind100 = sqrt(ua100m^2 + va100m^2),
    save one monthly NetCDF.
    """

    start_time, ua_path, va_path, out_path, force_refresh = task

    out_path = Path(out_path)

    if out_path.exists() and not force_refresh:
        return {
            "start_time": start_time,
            "out_path": str(out_path),
            "status": "exists",
            "error": None,
        }

    try:
        if out_path.exists() and force_refresh:
            out_path.unlink()

        with xr.open_dataset(ua_path, decode_times=True) as ds_u, \
             xr.open_dataset(va_path, decode_times=True) as ds_v:

            ua = ds_u["ua100m"]
            va = ds_v["va100m"]

            ua, va = xr.align(ua, va, join="inner")

            # Keep dimension order stable
            expected_dims = ["time", "realization", "lat", "lon"]
            ua = ua.transpose(*[d for d in expected_dims if d in ua.dims])
            va = va.transpose(*[d for d in expected_dims if d in va.dims])

            wind100 = np.hypot(ua, va).astype("float32")
            wind100 = wind100.rename("wind100")

            wind100.attrs = {
                "long_name": "100 m wind speed",
                "standard_name": "wind_speed",
                "units": "m s-1",
                "description": (
                    "Derived as sqrt(ua100m^2 + va100m^2) "
                    "from BARRA-RE2 daily ua100m and va100m."
                ),
            }

            ds_out = wind100.to_dataset()

            ds_out.attrs = {
                "title": "BARRA-RE2 daily 100 m wind speed",
                "source": "BARRA-RE2 AUS-22 daily ua100m and va100m",
                "derivation": "wind100 = sqrt(ua100m^2 + va100m^2)",
                "start_time": start_time,
            }

            n_time = ds_out.sizes["time"]
            n_realization = ds_out.sizes["realization"]
            n_lat = ds_out.sizes["lat"]
            n_lon = ds_out.sizes["lon"]

            encoding = {
                "wind100": {
                    "dtype": "float32",
                    "zlib": True,
                    "complevel": 4,
                    "shuffle": True,
                    "_FillValue": np.float32(np.nan),
                    "chunksizes": (
                        min(7, n_time),
                        1,
                        n_lat,
                        n_lon,
                    ),
                }
            }

            ds_out.to_netcdf(
                out_path,
                mode="w",
                engine="netcdf4",
                encoding=encoding,
            )

            ds_out.close()

        return {
            "start_time": start_time,
            "out_path": str(out_path),
            "status": "done",
            "error": None,
        }

    except Exception as e:
        return {
            "start_time": start_time,
            "out_path": str(out_path),
            "status": "failed",
            "error": str(e),
        }

In [11]:
%%time

tasks = [
    (
        row["start_time"],
        row["ua100m"],
        row["va100m"],
        row["out_path"],
        FORCE_REFRESH_MONTHLY,
    )
    for _, row in wind_pairs.iterrows()
]

monthly_results = []

with ProcessPoolExecutor(max_workers=MAX_WORKERS_WIND) as executor:
    futures = {
        executor.submit(process_month_to_netcdf, task): task
        for task in tasks
    }

    with tqdm(
        total=len(futures),
        desc="Processing monthly wind100 files",
        unit="month"
    ) as pbar:

        for future in as_completed(futures):
            result = future.result()
            monthly_results.append(result)

            if result["status"] == "failed":
                print(f"FAILED {result['start_time']}: {result['error']}")

            pbar.update(1)

monthly_results_df = pd.DataFrame(monthly_results).sort_values("start_time")
display(monthly_results_df["status"].value_counts())
display(monthly_results_df.head())

Processing monthly wind100 files: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 424/424 [04:41<00:00,  1.51month/s]


status
done    424
Name: count, dtype: int64

,start_time,out_path,status,error
4,1990-01,/scratch/ng72/ha2606/barra_re2_wind100_monthly...,done,None
0,1990-02,/scratch/ng72/ha2606/barra_re2_wind100_monthly...,done,None
11,1990-03,/scratch/ng72/ha2606/barra_re2_wind100_monthly...,done,None
5,1990-04,/scratch/ng72/ha2606/barra_re2_wind100_monthly...,done,None
30,1990-05,/scratch/ng72/ha2606/barra_re2_wind100_monthly...,done,None


CPU times: user 1.18 s, sys: 929 ms, total: 2.11 s
Wall time: 4min 42s


In [12]:
failed = monthly_results_df[monthly_results_df["status"] == "failed"]

print("Status counts:")
display(monthly_results_df["status"].value_counts())

if len(failed) > 0:
    display(failed)
    raise RuntimeError("Some monthly wind100 files failed. Fix these before continuing.")

monthly_files = sorted(MONTHLY_DIR.glob("wind100_*.nc"))

print(f"Monthly wind100 files found: {len(monthly_files)}")
print(monthly_files[0])
print(monthly_files[-1])

Status counts:


status
done    424
Name: count, dtype: int64

Monthly wind100 files found: 424
/scratch/ng72/ha2606/barra_re2_wind100_monthly_nc/wind100_1990-01.nc
/scratch/ng72/ha2606/barra_re2_wind100_monthly_nc/wind100_2025-04.nc
